# 05 – Query Expansion (Anime/Game Forum)

Notebook ini membuat kamus sinonim/istilah terkait untuk memperluas query. Kamus berisi lebih dari 15 pasangan. Query asli tetap dipertahankan, kemudian istilah alternatif ditambahkan. Setelah expansion, query diproses menggunakan `preprocess_text()` dari `preprocessing.py`, sama seperti preprocessing dokumen dan query retrieval.


In [1]:
import re
import pandas as pd


## 1. Fungsi preprocessing untuk query

Notebook ini mengimpor `preprocess_text()` dari `preprocessing.py`. Dengan begitu, query setelah query expansion menggunakan pipeline yang sama dengan dokumen korpus dan query pada sistem retrieval. Jangan membuat salinan fungsi preprocessing yang berbeda di notebook ini.


In [3]:
import sys
from pathlib import Path

# Mendukung notebook yang dijalankan dari root proyek maupun folder preprocessing.
_candidates = [
    Path.cwd() / "preprocessing",
    Path.cwd(),
    Path.cwd().parent / "preprocessing",
]
MODULE_DIR = next(
    (path for path in _candidates if (path / "preprocessing.py").exists()),
    None,
)
if MODULE_DIR is None:
    raise FileNotFoundError(
        "preprocessing.py tidak ditemukan. Pastikan folder preprocessing "
        "berada di root proyek."
    )

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from preprocessing import (
    ENGLISH_STOPWORDS,
    CUSTOM_STOPWORDS,
    STOPWORDS,
    clean_text,
    tokenize,
    remove_stopwords,
    stem_tokens,
    preprocess_text,
    preprocess_document,
)


## 2. Kamus sinonim dan istilah terkait

In [4]:
SYNONYM_GROUPS = [
    ("anime", "japanese animation"),
    ("manga", "japanese comic"),
    ("manhwa", "korean comic"),
    ("manhua", "chinese comic"),
    ("waifu", "favorite female character"),
    ("husbando", "favorite male character"),
    ("sub", "subtitles"),
    ("dub", "dubbing"),
    ("ova", "original video animation"),
    ("ost", "original soundtrack"),
    ("op", "opening song"),
    ("ed", "ending song"),
    ("isekai", "another world fantasy"),
    ("shonen", "boys manga"),
    ("rpg", "role playing game"),
    ("mmorpg", "massively multiplayer online role playing game"),
    ("fps", "first person shooter"),
    ("npc", "non player character"),
    ("pvp", "player versus player"),
    ("pve", "player versus environment"),
    ("dlc", "downloadable content"),
    ("bug", "glitch"),
    ("lag", "latency"),
    ("nerf", "weaken"),
    ("buff", "strengthen"),
    ("walkthrough", "strategy guide"),
    ("gacha", "random character draw"),
    ("guild", "clan")
]

def build_dictionary():
    dictionary = {}
    for left, right in SYNONYM_GROUPS:
        dictionary.setdefault(left.casefold(), set()).add(right.casefold())
        dictionary.setdefault(right.casefold(), set()).add(left.casefold())
    return dictionary

SYNONYM_DICTIONARY = build_dictionary()
print("Jumlah pasangan istilah:", len(SYNONYM_GROUPS))

Jumlah pasangan istilah: 28


In [5]:
pd.DataFrame(SYNONYM_GROUPS, columns=["Istilah", "Sinonim / istilah terkait"])

,Istilah,Sinonim / istilah terkait
0,anime,japanese animation
1,manga,japanese comic
2,manhwa,korean comic
3,manhua,chinese comic
4,waifu,favorite female character
5,husbando,favorite male character
6,sub,subtitles
7,dub,dubbing
8,ova,original video animation
9,ost,original soundtrack


## 3. Implementasi query expansion

Pencocokan istilah memperhatikan batas kata, mendukung frasa multi-kata, dan bersifat dua arah. Query asli tidak dihapus.

In [6]:
def expand_query(query: str, max_expansions: int = 10) -> str:
    original = query.strip()
    if not original:
        return original
    lowered = original.casefold()
    additions = []
    for term in sorted(SYNONYM_DICTIONARY, key=len, reverse=True):
        pattern = r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])"
        if re.search(pattern, lowered):
            for alternative in sorted(SYNONYM_DICTIONARY[term]):
                if alternative not in lowered and alternative not in additions:
                    additions.append(alternative)
                    if len(additions) >= max_expansions:
                        break
        if len(additions) >= max_expansions:
            break
    return original if not additions else original + " " + " ".join(additions)

def expand_and_preprocess_query(query: str) -> str:
    return preprocess_text(expand_query(query))

query = "best RPG game"
print("Query awal   :", query)
print("Query hasil  :", expand_query(query))
print("Siap diproses:", expand_and_preprocess_query(query))

Query awal   : best RPG game
Query hasil  : best RPG game role playing game
Siap diproses: best rpg game role play game


## 4. Pengujian minimal lima query

In [7]:
test_queries = [
    "anime recommendation",
    "manga with good story",
    "best RPG game",
    "how to fix lag in FPS",
    "anime sub and dub",
]
rows = []
for q in test_queries:
    rows.append({
        "Query awal": q,
        "Setelah expansion": expand_query(q),
        "Expansion + preprocessing": expand_and_preprocess_query(q)
    })
pd.DataFrame(rows)

,Query awal,Setelah expansion,Expansion + preprocessing
0,anime recommendation,anime recommendation japanese animation,anim recommend japanes anim
1,manga with good story,manga with good story japanese comic,manga good stori japanes comic
2,best RPG game,best RPG game role playing game,best rpg game role play game
3,how to fix lag in FPS,how to fix lag in FPS first person shooter lat...,fix lag fps first person shooter latenc
4,anime sub and dub,anime sub and dub japanese animation subtitles...,anim sub dub japanes anim subtitl dub
